step 1: copy cells 1-3 from 02

In [5]:
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import case, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR, TLS224_APPLN_CPC,
)

patstat = PatstatClient(env="TEST")
db = patstat.orm()

# ---- sector definition
SECTOR_NAME = "Quantum computing"
SECTOR_CPC_CODES = ("G06N10/", "H04L9/0852", "H04L9/0855", "H04L9/0858", "H04B10/70")
SECTOR_KEYWORDS = (
    "quantum comput", "qubit", "quantum processor", "quantum algorithm",
    "quantum circuit", "quantum gate", "quantum annealing", "quantum annealer",
    "quantum error correction",
)


# ---- helper filters
def code_filter(column, codes):
    clean = func.replace(column, " ", "")
    return or_(*[clean.like(f"{c}%") for c in codes])


def keyword_filter(terms):
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(*[title.like(f"%{t}%") for t in terms], *[abstract.like(f"%{t}%") for t in terms])


# ---- find the quantum families
family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
q_cpc = (
    db.query(family_id)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .filter(code_filter(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_CODES))
)
q_kw = (
    db.query(family_id)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(keyword_filter(SECTOR_KEYWORDS))
)
sector_families = set(patstat.df(q_cpc.union(q_kw))["docdb_family_id"])
print(len(sector_families), "quantum families")


30 quantum families


whats added: appln_id <900000000 to remove dumies. nb_citing_docdb_fam is how many later inventions cite
cell 2:

In [6]:
df = patstat.df(
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_kind,              # new
        TLS201_APPLN.ipr_type,                # new
        TLS201_APPLN.earliest_filing_year,
        TLS201_APPLN.granted,
        TLS201_APPLN.docdb_family_size,
        TLS201_APPLN.nb_citing_docdb_fam,     # new
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(list(sector_families)),
        TLS201_APPLN.appln_id < 900000000,    # new: no "dummy" applications
    )
)
print(len(df), "applications in", df["docdb_family_id"].nunique(), "families")
df[["ipr_type", "appln_kind"]].value_counts()


46 applications in 30 families


ipr_type  appln_kind
PI        A             36
          W              6
UM        U              4
Name: count, dtype: int64

cell 3: one row per family

In [7]:
fam = df[df["ipr_type"] == "PI"].groupby("docdb_family_id").agg(
    year=("earliest_filing_year", "min"),
    family_size=("docdb_family_size", "max"),
    granted=("granted", lambda s: (s == "Y").any()),
    citations=("nb_citing_docdb_fam", "max"),
)

print("granted (share of families):", round(fam["granted"].mean(), 2))
fam[["family_size", "citations"]].describe().round(1)


granted (share of families): 0.6


,family_size,citations
count,30.0,30.0
mean,2.8,12.5
std,4.1,30.1
min,1.0,0.0
25%,1.0,0.0
50%,2.0,3.0
75%,2.0,9.0
max,23.0,159.0


4

In [8]:
from epo.tipdata.patstat.database.models import TLS206_PERSON, TLS207_PERS_APPLN

people = patstat.df(
    db.query(
        TLS207_PERS_APPLN.appln_id,
        TLS206_PERSON.person_id,
        TLS206_PERSON.person_name,
        TLS206_PERSON.psn_name,
        TLS206_PERSON.han_name,
        TLS206_PERSON.psn_sector,
        TLS206_PERSON.person_ctry_code,
    )
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        TLS207_PERS_APPLN.appln_id.in_(df["appln_id"].tolist()),
        TLS207_PERS_APPLN.applt_seq_nr > 0,          # applicants only
    )
)
people = people.merge(df[["appln_id", "docdb_family_id", "appln_auth"]], on="appln_id")

print("different person_id:  ", people["person_id"].nunique())
print("different person_name:", people["person_name"].nunique())
print("different psn_name:   ", people["psn_name"].nunique())
print("different han_name:   ", people["han_name"].nunique())

pd.set_option("display.max_colwidth", 60)
people[["person_name", "psn_name", "han_name", "psn_sector", "person_ctry_code"]].drop_duplicates().sort_values("psn_name")


different person_id:   30
different person_name: 29
different psn_name:    28
different han_name:    26


,person_name,psn_name,han_name,psn_sector,person_ctry_code
6,1QB INFORMATION TECHNOLOGIES INC.,1QB INFORMATION TECHNOLOGIES,1QB INFORMATION TECH INC,COMPANY,CA
41,CHONGQING UNIVERSITY,CHONGQING UNIVERSITY,CHONGQING UNIVERSITY,UNIVERSITY,
51,FERRY DAVID K.,FERRY DAVID K.,FERRY DAVID K.,None,US
49,GILBERT MATTHEW J.,GILBERT MATTHEW J.,GILBERT MATTHEW J.,None,US
19,GUANGXI UNIVERSITY,GUANGXI UNIVERSITY,GUANGXI UNIVERSITY,UNIVERSITY,
40,"HEFEI ORIGIN QUANTUM COMPUTING TECHNOLOGY CO., LTD.","HEFEI ORIGIN QUANTUM COMPUTING TECHNOLOGY CO., LTD.",ORIGIN QUANTUM COMPUTING TECH CO LTD,UNKNOWN,CN
12,"Hitachi, Ltd.",HITACHI,HITACHI LTD,COMPANY,JP
8,International Business Machines Corporation,IBM (INTERNATIONAL BUSINESS MACHINES CORPORATION),IBM CORP,COMPANY,US
47,Mitsubishi Electric Corporation,MITSUBISHI ELECTRIC CORPORATION,MITSUBISHI ELECT CORP,COMPANY,JP
45,"NANNING CRRC RAIL TRANSIT EQUIPMENT CO., LTD.",NANNING CRRC RAIL TRANSIT EQUIPMENT COMPANY,"NANNING CRRC RAIL TRANSIT EQUIPMENT CO., LTD.",COMPANY,
